# 01 — Explore the Mobility Database GTFS Catalog

This notebook helps you explore the publicly available GTFS feeds from the
[Mobility Database](https://mobilitydatabase.org/). Use it to understand what's
available before selecting feeds for the UmarTransit-1B training dataset.

**Data source:** `https://bit.ly/catalogs-csv` (Mobility Database CSV catalog)

**License:** Most public GTFS feeds are public domain or CC-BY — compatible with ML training.

In [ ]:
# Install dependencies (for Google Colab)
# Uncomment the line below if running in Colab
# !pip install pandas requests tqdm matplotlib -q

In [ ]:
import pandas as pd
import requests
import matplotlib.pyplot as plt

# Download the Mobility Database catalog
CATALOG_URL = "https://bit.ly/catalogs-csv"
print("Downloading Mobility Database catalog...")
df = pd.read_csv(CATALOG_URL, low_memory=False)
print(f"Total entries: {len(df):,}")
print(f"Columns: {list(df.columns)}")

## Basic Statistics

In [ ]:
# Filter to active GTFS feeds with direct download URLs
gtfs = df.copy()
if "data_type" in gtfs.columns:
    gtfs = gtfs[gtfs["data_type"] == "gtfs"]
if "status" in gtfs.columns:
    gtfs = gtfs[gtfs["status"] == "active"]
if "urls.direct_download" in gtfs.columns:
    gtfs = gtfs[gtfs["urls.direct_download"].notna()]
if "urls.authentication_type" in gtfs.columns:
    gtfs = gtfs[gtfs["urls.authentication_type"].isna() | (gtfs["urls.authentication_type"] == "")]

print(f"Active GTFS feeds with direct download (no auth): {len(gtfs):,}")
print(f"\nData types: {df['data_type'].value_counts().to_dict() if 'data_type' in df.columns else 'N/A'}")
print(f"\nStatus breakdown: {df['status'].value_counts().to_dict() if 'status' in df.columns else 'N/A'}")

## Feeds by Country (Top 20)

In [ ]:
# Top 20 countries by number of GTFS feeds
if "location.country_code" in gtfs.columns:
    country_counts = gtfs["location.country_code"].value_counts().head(20)

    fig, ax = plt.subplots(figsize=(12, 6))
    country_counts.plot(kind="bar", ax=ax, color="steelblue")
    ax.set_title("Top 20 Countries by Number of Active GTFS Feeds")
    ax.set_xlabel("Country Code")
    ax.set_ylabel("Number of Feeds")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()

    print(f"\nTotal countries represented: {gtfs['location.country_code'].nunique()}")
    print(f"\nTop 20:\n{country_counts.to_string()}")

## Browse Feeds by Provider

Search for specific transit agencies to find their feed IDs and download URLs.

In [ ]:
# Search for a specific provider (change the search term as needed)
search_term = "MTA"

if "provider" in gtfs.columns:
    matches = gtfs[gtfs["provider"].str.contains(search_term, case=False, na=False)]
    display_cols = ["mdb_source_id", "provider", "location.country_code",
                    "location.subdivision_name", "location.municipality",
                    "urls.direct_download", "urls.license"]
    available = [c for c in display_cols if c in matches.columns]
    print(f"Found {len(matches)} feeds matching '{search_term}':\n")
    if not matches.empty:
        print(matches[available].to_string(index=False))
    else:
        print("No matches found. Try a different search term.")

## Sample Feeds from Different Regions

A quick look at feed diversity across continents.

In [ ]:
# Sample feeds from different regions
regions = {
    "North America": ["US", "CA", "MX"],
    "Europe": ["GB", "DE", "FR", "NL", "BE", "CH"],
    "Asia-Pacific": ["AU", "NZ", "JP", "KR", "IN"],
    "South America": ["BR", "CL", "AR"],
}

if "location.country_code" in gtfs.columns:
    for region, codes in regions.items():
        region_feeds = gtfs[gtfs["location.country_code"].isin(codes)]
        print(f"\n{'='*50}")
        print(f"{region}: {len(region_feeds)} feeds")
        print(f"{'='*50}")
        if not region_feeds.empty:
            sample = region_feeds.sample(min(3, len(region_feeds)), random_state=42)
            for _, row in sample.iterrows():
                print(f"  - {row.get('provider', 'N/A')} "
                      f"({row.get('location.country_code', '')}, "
                      f"{row.get('location.municipality', 'N/A')})")